In [1]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Layer
from tensorflow.keras.layers import (Reshape,Conv2DTranspose,Add,Conv2D,MaxPool2D,Dense,Flatten,InputLayer,BatchNormalization,Input)
from tensorflow.keras.optimizers import Adam

In [2]:
(x_train,_),(x_test,_)=tf.keras.datasets.mnist.load_data()
mnist_digits=np.concatenate([x_train,x_test],axis=0)
mnist_digits=np.expand_dims(mnist_digits,-1).astype("float32")/255

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [3]:
dataset=tf.data.Dataset.from_tensor_slices(mnist_digits)

In [4]:
dataset

<_TensorSliceDataset element_spec=TensorSpec(shape=(28, 28, 1), dtype=tf.float32, name=None)>

In [5]:
BATCH_SIZE=128
LATENT_DIM=2

In [6]:
train_dataset=(
    dataset
    .shuffle(buffer_size=1024,reshuffle_each_iteration=True)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [7]:
train_dataset

<_PrefetchDataset element_spec=TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name=None)>

# **MODELING**

# **sampling**

In [8]:
class Sampling(Layer):
  def call(self,inputs):
    mean,log_var=inputs
    return mean+tf.exp(0.5*log_var)*tf.random.normal(shape=(tf.shape(mean)[0],tf.shape(mean)[1]))

# **ENCODER**

In [10]:
encoder_inputs=Input(shape=(28,28,1))
x=Conv2D(32,3,activation='relu',strides=2,padding='same')(encoder_inputs)
x=Conv2D(64,3,activation='relu',strides=2,padding='same')(x)

x=Flatten()(x)
x=Dense(16,activation='relu')(x)

mean=Dense(LATENT_DIM,)(x)
log_var=Dense(LATENT_DIM,)(x)

z=Sampling()([mean,log_var])

encoder_model=Model(encoder_inputs,[z,mean,log_var],name='encoder')
encoder_model.summary()

Model: "encoder"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 28, 28, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 14, 14,    │        320 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 7, 7, 64)  │     18,496 │ conv2d_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 3136)      │          0 │ conv2d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 16)        │     50,192 │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 2)         │         34 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 2)         │         34 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sampling (Sampling) │ (None, 2)         │          0 │ dense_4[0][0],    │
│                     │                   │            │ dense_5[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 69,076 (269.83 KB)

 Trainable params: 69,076 (269.83 KB)

 Non-trainable params: 0 (0.00 B)

# **DECODER**

In [12]:
latent_inputs=Input(shape=(LATENT_DIM,))

x=Dense(7*7*64,activation='relu')(latent_inputs)
x=Reshape((7,7,64))(x)

x=Conv2DTranspose(64,3,activation='relu',strides=2,padding='same')(x)
x=Conv2DTranspose(32,3,activation='relu',strides=2,padding='same')(x)

decoder_output=Conv2DTranspose(1,3,activation='sigmoid',padding='same')(x)
decoder_model=Model(latent_inputs,decoder_output,name='decoder')
decoder_model.summary()

Model: "decoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 2)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 3136)           │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_1 (Reshape)             │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_1              │ (None, 14, 14, 64)     │        36,928 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_2              │ (None, 28, 28, 32)     │        18,464 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_3              │ (None, 28, 28, 1)      │           289 │
│ (Conv2DTranspose)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 65,089 (254.25 KB)

 Trainable params: 65,089 (254.25 KB)

 Non-trainable params: 0 (0.00 B)

# **VAE MODEL**

In [14]:
vae_input=Input(shape=(28,28,1),name='vae_input')

z,_,_=encoder_model(vae_input)
output=decoder_model(z)
vae=Model(vae_input,output,name='vae')
vae.summary()

Model: "vae"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vae_input (InputLayer)          │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder (Functional)            │ [(None, 2), (None, 2), │        69,076 │
│                                 │ (None, 2)]             │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder (Functional)            │ (None, 28, 28, 1)      │        65,089 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 134,165 (524.08 KB)

 Trainable params: 134,165 (524.08 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
for i in range(3):
  print(vae.layers[i])

<InputLayer name=vae_input, built=True>
<Functional name=encoder, built=True>
<Functional name=decoder, built=True>


# **TRAINING**

In [16]:
OPTIMIZER=Adam(learning_rate=1e-3)
EPOCHS=20

In [21]:
def custom_loss(y_true,y_pred,mean,log_var):
  loss_rec=tf.reduce_mean(tf.reduce_sum(tf.keras.losses.binary_crossentropy(y_true,y_pred),axis=(1,2)))
  loss_reg=-0.5*(1 + log_var - tf.square(mean) - tf.exp(log_var))

  return loss_rec+tf.reduce_mean(tf.reduce_sum(loss_reg,axis=1))

In [22]:
@tf.function
def training_block(x_batch):
  with tf.GradientTape() as recorder:
    z,mean,log_var=encoder_model(x_batch)
    y_pred=decoder_model(z)
    y_true=x_batch
    loss=custom_loss(y_true,y_pred,mean,log_var)

  partial_derivatives=recorder.gradient(loss,vae.trainable_weights)
  OPTIMIZER.apply_gradients(zip(partial_derivatives,vae.trainable_weights))
  return loss

In [23]:
def neuralearn(epochs):
  for epoch in range(1,epochs+1):
    print('Training starts for epoch number {}'.format(epoch))

    for step,x_batch in enumerate(train_dataset):
      loss=training_block(x_batch)
    print('Training loss is :',loss)
  print('Training Complete')

In [ ]:
neuralearn(EPOCHS)

Training starts for epoch number 1
Training loss is : tf.Tensor(190.38776, shape=(), dtype=float32)
Training starts for epoch number 2
Training loss is : tf.Tensor(175.74023, shape=(), dtype=float32)
Training starts for epoch number 3
Training loss is : tf.Tensor(165.6305, shape=(), dtype=float32)
Training starts for epoch number 4
Training loss is : tf.Tensor(161.19366, shape=(), dtype=float32)
Training starts for epoch number 5
Training loss is : tf.Tensor(155.4922, shape=(), dtype=float32)
Training starts for epoch number 6
Training loss is : tf.Tensor(169.95795, shape=(), dtype=float32)
Training starts for epoch number 7
Training loss is : tf.Tensor(155.73808, shape=(), dtype=float32)
Training starts for epoch number 8
Training loss is : tf.Tensor(153.09851, shape=(), dtype=float32)
Training starts for epoch number 9
Training loss is : tf.Tensor(156.33861, shape=(), dtype=float32)
Training starts for epoch number 10
Training loss is : tf.Tensor(152.05968, shape=(), dtype=float32)
T

# **OVERRIDING TRAIN_STEP METHOD**

In [ ]:
class VAE(tf.keras.Model):
  def __init__(self,encoder_model,decoder_model):
    super(VAE,self).__init__()
    self.encoder=encoder_model
    self.decoder=decoder_model
    self.loss_tracker=tf.keras.metrics.Mean(name='loss')

  @property
  def metrics(self):
    return [self.loss_tracker]

  def train_step(self,x_batch):
    with tf.GradientTape() as recorder:
      z,mean,log_var=self.encoder(x_batch)
      y_pred=self.decoder(z)
      y_true=x_batch
      loss=custom_loss(y_true,y_pred,mean,log_var)

    partial_derivatives=recorder.gradient(loss,self.trainable_weights)
    OPTIMIZER.apply_gradients(zip(partial_derivatives,self.trainable_weights))

    self.loss_tracker.update_state(loss)
    return{'loss':self.loss_tracker.result()}



In [ ]:
model=VAE(encoder_model,decoder_model)
model.compile(optimizer=OPTIMIZER)
model.fit(train_dataset,epochs=20,batch_size=128)

# **TESTING**

In [ ]:
scale=1
n=16

In [ ]:
grid_x=np.linspace(-scale,scale,16)
grid_y=np.linspace(-scale,scale,16)

In [ ]:
print(grid_x,grid_y)

In [ ]:
plt.figure(fig_size=(12,12))
k=0
for i in grid_x:
  for j in grid_y:
    ax=plt.subplot(n,n,k+1)
    input=tf.constant([[i,j]])
    out=model.decoder.predict(input)[0][...,0]
    plt.imshow(out,cmap='greys_r')
    plt.axis('off')
    k+=1

In [ ]:
print(vae.layers[2].predict(tf.constant([[-1,1]]))[0][...,0].shape)

In [ ]:
(x_train,y_train),_=tf.keras.datasets.mnist.load_data()
mnist_digits=np.expand_dims(x_train,-1).astype("float32")/255

In [ ]:
z,_,_=vae.layers[1].predict(x_train)
plt.figure(figsize=(12,12))
plt.scatter(z[:,0],z[:1],c=y_train)
plt.colorbar()
plt.show()